# Grading INFO 4617 from Canvas

This notebook grades the course's recurring work by reading from and posting to Canvas:
**attendance**, **Textbook Revisions** (weeks 2–9), the **portfolio checkpoints** (weeks 11–14),
and **Notebook Labs**. The rules are in [`docs/grading.md`](../docs/grading.md).

Run it top to bottom. Each section builds a table of proposed scores and comments and saves it as
a CSV in `GRADING_DIR`. Open the CSV, change anything you disagree with, and set `review` to
`False` on each row you have settled. The last section posts one table to Canvas, but only with
`APPLY = True`, only when no row still needs review, and only after you type how many rows you're
posting. It then reads every grade and comment back from Canvas.

**Clear the outputs before you commit this notebook**: they show students' names and scores, and
the repository is public. Student data lives only in `GRADING_DIR`, outside the repository.

## Setup

Set `CANVAS_TOKEN` in your environment (Canvas → Account → Settings → New Access Token).
`GITHUB_TOKEN` is optional: without it GitHub allows 60 requests an hour, which isn't enough for
a week of revisions. A read-only fine-grained token with no repository access is enough.

The rules from `docs/grading.md` that the code needs are set here. When that file changes,
change them too.

In [ ]:
import json
import math
import os
import re
import subprocess
import sys
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

sys.path.insert(0, ".")
import course_spec as spec

APPLY = False     # True only when you are ready to post the table in the last section

CANVAS = f"https://canvas.colorado.edu/api/v1/courses/{spec.NEW_COURSE_ID}"
GRADING_DIR = Path(os.environ.get("GRADING_DIR", Path.home() / "info4617-grading"))
GRADING_DIR.mkdir(parents=True, exist_ok=True)

canvas = requests.Session()
canvas.headers["Authorization"] = "Bearer " + os.environ["CANVAS_TOKEN"].strip()
github = requests.Session()
github.headers["Accept"] = "application/vnd.github+json"
if os.environ.get("GITHUB_TOKEN"):
    github.headers["Authorization"] = "Bearer " + os.environ["GITHUB_TOKEN"].strip()
gh_cache = {}

# docs/grading.md
CLASS_REPOS = {"cuinfoscience/web-data-science-book", "cuinfoscience/info4617-fall2026", "cuinfoscience/info-missing-manual"}
OPTIONAL_STEPS = {5: {2}, 7: {4, 5}}       # lab steps the instructor told students to skip
LATE_GRACE_HOURS = 24
LATE_CAP = 5
COMMENTS = {
    "own_comment": "Full credit this week, but next time please submit the link to your own comment (click its timestamp to copy a URL ending in #issuecomment-…) rather than the link to the whole issue.",
    "blank": "This upload was the blank exercise scaffold, so please save your notebook after completing the exercises, export it to HTML, and upload that completed file.",
    "unrun": "Please run every cell before exporting so that both your code and its output appear in the HTML you submit.",
    "traceback": "When you hit a traceback, check that the libraries you need are imported and that the cells defining your variables have been run first, in order.",
    "unseen": "Your interpretation reports results that don't appear in your notebook's output, so please run every cell before exporting so the evidence for your claims is visible.",
}
COLUMNS = ["kind", "week", "assignment_id", "user_id", "name", "action", "score", "rubric", "comment", "review", "notes", "url"]

print("Student data goes to", GRADING_DIR)
print("Posting is", "ON" if APPLY else "off (dry run)")

## The roster and the assignments

Every later section uses these two tables. Canvas returns long lists a page at a time, so each
request follows the `next` link until there isn't one.

In [ ]:
url, params, users = f"{CANVAS}/users", {"enrollment_type[]": "student", "per_page": 100}, []
while url:
    r = canvas.get(url, params=params)
    r.raise_for_status()
    users += r.json()
    url, params = r.links.get("next", {}).get("url"), None

roster = pd.DataFrame({
    "user_id": [u["id"] for u in users],
    "name": [u["name"] for u in users],
    "identikey": [(u.get("login_id") or "").lower() for u in users],
}).set_index("user_id")

url, params, groups = f"{CANVAS}/assignment_groups", {"include[]": "assignments", "per_page": 100}, []
while url:
    r = canvas.get(url, params=params)
    r.raise_for_status()
    groups += r.json()
    url, params = r.links.get("next", {}).get("url"), None

rows = []
for g in groups:
    for a in g["assignments"]:
        m = re.match(r"Week (\d+)", a["name"])
        rows.append({"assignment_id": a["id"], "name": a["name"], "group": g["name"],
                     "week": int(m.group(1)) if m else None, "points": a["points_possible"],
                     "due_at": a.get("due_at"), "published": a["published"],
                     "has_rubric": bool(a.get("rubric"))})
assignments = pd.DataFrame(rows)
print(len(roster), "students,", len(assignments), "assignments")

## What's left to grade

Submissions in published assignments that have no score, or were resubmitted after they were
graded.

In [ ]:
left = []
for _, a in assignments[assignments["published"] & assignments["group"].isin(["Notebook Labs", "Textbook Revisions", "Final Project"])].iterrows():
    url, params, subs = f"{CANVAS}/assignments/{a['assignment_id']}/submissions", {"include[]": "submission_comments", "per_page": 100}, []
    while url:
        r = canvas.get(url, params=params)
        r.raise_for_status()
        subs += r.json()
        url, params = r.links.get("next", {}).get("url"), None
    for s in subs:
        if s["user_id"] not in roster.index or not s.get("submitted_at"):
            continue
        resubmitted = bool(s.get("graded_at")) and s["submitted_at"] > s["graded_at"]
        if s.get("score") is None or resubmitted:
            last = (s.get("submission_comments") or [{}])[-1].get("comment", "")
            left.append({"assignment": a["name"], "name": roster.loc[s["user_id"], "name"], "score": s.get("score"),
                         "submitted_at": s["submitted_at"][:16], "resubmitted_after_grading": resubmitted,
                         "last_comment": last[:90]})
pd.DataFrame(left)

## Attendance

The sign-in sheet is a CSV with one row per sign-in: the student's **identikey** and a
**timestamp**, in Mountain time unless the timestamp says otherwise. Change the column names if
your export uses others.

A week's score is **one point per session held**. A session is a scheduled meeting day that has
sign-ins, so a cancelled class drops out. Sign-ins on other days are listed and ignored. Add a
make-up session's date to `EXTRA_DATES` to count it.

In [ ]:
WEEK = 11
SIGNINS = GRADING_DIR / "attendance.csv"
ID_COLUMN, TIME_COLUMN = "identikey", "timestamp"
EXTRA_DATES = []          # e.g. [date(2026, 11, 7)] for a make-up session

sheet = pd.read_csv(SIGNINS, dtype=str)
times = pd.to_datetime(sheet[TIME_COLUMN], errors="coerce")
if times.dt.tz is None:
    times = times.dt.tz_localize(spec.TZ, ambiguous="NaT", nonexistent="NaT")
sheet["identikey"] = sheet[ID_COLUMN].str.strip().str.lower()
sheet["date"] = times.dt.tz_convert(spec.TZ).dt.date
sheet = sheet.dropna(subset=["date"])
sheet["week"] = [(d - spec.week_monday(1)).days // 7 + 1 for d in sheet["date"]]

this_week = sheet[sheet["week"] == WEEK]
meeting_days = set(spec.meeting_days(WEEK)) | set(EXTRA_DATES)
for d in sorted(set(this_week["date"]) - meeting_days):
    print(f"{d}: {(this_week['date'] == d).sum()} sign-ins on a day the class doesn't meet, ignored")
this_week = this_week[this_week["date"].isin(meeting_days)]
sessions = sorted(set(this_week["date"]))
attended = this_week.drop_duplicates(["identikey", "date"]).groupby("identikey")["date"].nunique()

print("Sessions held:", len(sessions), sessions)
print("Identikeys not on the roster:", sorted(set(attended.index) - set(roster["identikey"])))

a = assignments[assignments["name"] == f"Week {WEEK:02d} Attendance"].iloc[0]
print(f"{a['name']} is worth {a['points']:g} points in Canvas")

The assignment should be worth one point per session held. This changes its points, but only while
the week has no grades: changing points after grading changes students' percentages.

In [ ]:
r = canvas.get(f"{CANVAS}/assignments/{a['assignment_id']}/submissions", params={"per_page": 100})
r.raise_for_status()
graded = any(s.get("score") is not None for s in r.json())
if a["points"] != len(sessions):
    if graded:
        print(f"{a['name']} already has grades: leave it at {a['points']:g} points")
    elif APPLY:
        r = canvas.put(f"{CANVAS}/assignments/{a['assignment_id']}", data={"assignment[points_possible]": len(sessions)})
        r.raise_for_status()
        print(f"{a['name']} is now worth {len(sessions)} points")
    else:
        print(f"With APPLY = True this sets {a['name']} to {len(sessions)} points")

In [ ]:
rows = []
for user_id, st in roster.iterrows():
    n = int(attended.get(st["identikey"], 0))
    rows.append({"kind": "attendance", "week": WEEK, "assignment_id": a["assignment_id"], "user_id": user_id,
                 "name": st["name"], "action": "grade", "score": n, "rubric": "", "comment": "",
                 "review": False, "notes": f"{n} of {len(sessions)} sessions", "url": ""})
attendance = pd.DataFrame(rows, columns=COLUMNS)
path = GRADING_DIR / f"attendance-w{WEEK:02d}.csv"
attendance.to_csv(path, index=False)
print("Review and edit:", path)
attendance["score"].value_counts().sort_index()

## Textbook Revisions, weeks 2–9

One point for any issue, pull request, or comment the student made on GitHub in a class repository.
A commit, a compare page, or a pull request inside their own fork counts too.

First, every submitted link is looked up on GitHub: who made it, when, how long it is, and who else
commented on it or reviewed it.

In [ ]:
revision_assignments = assignments[(assignments["group"] == "Textbook Revisions") & assignments["week"].between(2, 9)]
subs = []
for _, a in revision_assignments.iterrows():
    url, params = f"{CANVAS}/assignments/{a['assignment_id']}/submissions", {"per_page": 100}
    while url:
        r = canvas.get(url, params=params)
        r.raise_for_status()
        for s in r.json():
            if s["user_id"] in roster.index:
                subs.append({"week": a["week"], "assignment_id": a["assignment_id"], "user_id": s["user_id"],
                             "url": (s.get("url") or "").strip(), "submitted": s["workflow_state"] != "unsubmitted"})
        url, params = r.links.get("next", {}).get("url"), None
subs = pd.DataFrame(subs)

links = {}
for link in sorted(set(subs["url"]) - {""}):
    m = re.match(r"https?://github\.com/([^/]+)/([^/#?]+)(?:/(issues|pull|commit|compare|blob|tree)/([^#?]+))?(?:\?[^#]*)?(?:#(.*))?", link)
    if not m:
        links[link] = {"type": "not github"}
        continue
    owner, repo, kind, rest, anchor = m.groups()
    d = {"owner": owner, "repo": repo, "class_repo": f"{owner}/{repo}".lower() in CLASS_REPOS}
    base = f"https://api.github.com/repos/{owner}/{repo}"
    if kind in ("issues", "pull"):
        d["number"] = int(rest.split("/")[0])
        d["type"] = "pr" if kind == "pull" else "issue"
        if anchor and anchor.startswith("issuecomment-"):
            d["type"], d["comment_id"] = "comment", int(anchor.split("-")[1])
        if anchor and anchor.startswith("pullrequestreview-"):
            d["type"], d["review_id"] = "review", int(anchor.split("-")[1])
        parts = rest.split("/")
        if kind == "pull" and len(parts) > 2 and parts[1] == "changes" and re.fullmatch(r"[0-9a-f]{7,40}", parts[2]):
            d["type"], d["sha"] = "commit", parts[2]
    elif kind == "commit":
        d["type"], d["sha"] = "commit", rest.split("/")[0]
    elif kind == "compare":
        d["type"], d["compare"] = "compare", rest
    else:
        d["type"] = "file" if kind else "repo"

    # look it up: the item, then the comment or review the link points at
    if "number" in d:
        r = github.get(f"{base}/issues/{d['number']}")
        item = r.json() if r.ok else {}
        d["author"], d["created_at"], d["size"] = (item.get("user") or {}).get("login"), item.get("created_at"), len(item.get("body") or "")
        people, page = [], 1
        while True:
            r = github.get(f"{base}/issues/{d['number']}/comments", params={"per_page": 100, "page": page})
            batch = r.json() if r.ok else []
            if not batch:
                break
            people += [(c.get("user") or {}).get("login") for c in batch]
            page += 1
        if item.get("pull_request"):
            r = github.get(f"{base}/pulls/{d['number']}/reviews", params={"per_page": 100})
            people += [(c.get("user") or {}).get("login") for c in (r.json() if r.ok else [])]
        d["participants"] = sorted({p for p in people if p})
    if d["type"] == "comment":
        r = github.get(f"{base}/issues/comments/{d['comment_id']}")
        c = r.json() if r.ok else {}
        d["author"], d["created_at"], d["size"] = (c.get("user") or {}).get("login"), c.get("created_at"), len(c.get("body") or "")
    if d["type"] == "review":
        r = github.get(f"{base}/pulls/{d['number']}/reviews/{d['review_id']}")
        c = r.json() if r.ok else {}
        d["author"], d["created_at"], d["size"] = (c.get("user") or {}).get("login"), c.get("submitted_at"), len(c.get("body") or "")
    if d["type"] == "commit":
        r = github.get(f"{base}/commits/{d['sha']}")
        c = r.json() if r.ok else {}
        d["author"], d["created_at"] = (c.get("author") or {}).get("login"), ((c.get("commit") or {}).get("author") or {}).get("date")
    if d["type"] == "compare":
        r = github.get(f"{base}/compare/{d['compare']}")
        commits = (r.json() if r.ok else {}).get("commits") or [{}]
        d["author"], d["created_at"] = (commits[-1].get("author") or {}).get("login"), ((commits[-1].get("commit") or {}).get("author") or {}).get("date")
    if d["type"] in ("file", "repo"):
        d["author"] = owner
    links[link] = d
print(len(links), "links looked up")

Each student's GitHub account is the usual author of their links. `other_logins` lists any other
author. If one of those is the same student's second account, add it to `LOGINS` and run this cell
and the next one again.

In [ ]:
LOGINS = {}   # Canvas user_id -> confirmed GitHub logins, e.g. {123456: ["main-login", "second-login"]}

votes = defaultdict(Counter)
for _, s in subs[subs["url"] != ""].iterrows():
    if links[s["url"]].get("author"):
        votes[s["user_id"]][links[s["url"]]["author"]] += 1
mine = {}
for user_id in roster.index:
    mine[user_id] = set(LOGINS.get(user_id, [])) or ({votes[user_id].most_common(1)[0][0]} if votes[user_id] else set())
mapping = pd.DataFrame({
    "name": roster["name"],
    "identikey": roster["identikey"],
    "github": [", ".join(sorted(mine[u])) for u in roster.index],
    "other_logins": [", ".join(k for k in votes[u] if k not in mine[u]) for u in roster.index],
})
mapping[mapping["other_logins"] != ""]

The scores:
- **Someone else's item.** If the student commented on it, they get credit plus the reminder to link
  their own comment. If not, the row is 0 and flagged.
- **The same item in two weeks** earns credit once, in the week it was made. The other weeks are 0.
- **Flagged:** short comments, links that aren't on GitHub, and work in other repositories.

In [ ]:
rows, seen = [], defaultdict(list)
for _, s in subs.sort_values("week").iterrows():
    row = {"kind": "revision", "week": int(s["week"]), "assignment_id": s["assignment_id"], "user_id": s["user_id"],
           "name": roster.loc[s["user_id"], "name"], "action": "grade", "score": 1, "rubric": "", "comment": "",
           "review": False, "notes": "", "url": s["url"]}
    if not s["submitted"] or not s["url"]:
        row.update(score=0, notes="no submission")
        rows.append(row)
        continue
    d, me, notes = links[s["url"]], mine[s["user_id"]], [links[s["url"]]["type"]]
    if d["type"] == "not github":
        row.update(score=0, review=True)
        notes.append("not a GitHub link")
    elif d.get("author") not in me:
        if me & set(d.get("participants", [])):
            row["comment"] = COMMENTS["own_comment"]
            notes.append(f"linked {d.get('author')}'s item; the student commented on it")
        else:
            row.update(score=0, review=True)
            notes.append(f"item is by {d.get('author')}, not {', '.join(sorted(me)) or 'a known login'}")
    if d["type"] in ("commit", "compare", "file", "repo"):
        notes.append("not an issue, PR, or comment (counts if it's the student's)")
    if d["type"] != "not github" and not d["class_repo"]:
        notes.append(f"in {d['owner']}/{d['repo']}")
    if d["type"] in ("comment", "review") and (d.get("size") or 0) < 80:
        row["review"] = True
        notes.append(f"short ({d.get('size')} characters)")
    row["notes"] = "; ".join(notes)
    # the same item: a comment, a review, a commit, an issue or PR, or (for files and compare pages) the link
    item = d.get("comment_id") or d.get("review_id") or d.get("sha") or d.get("number") or re.sub(r"[#?].*$", "", s["url"]).lower()
    seen[(s["user_id"], f"{d.get('owner', '')}/{d.get('repo', '')}".lower(), item)].append((len(rows), d.get("created_at")))
    rows.append(row)

for key, found in seen.items():
    if len(found) < 2:
        continue
    created = found[0][1]
    made = (datetime.fromisoformat(created.replace("Z", "+00:00")).astimezone(spec.TZ).date() - spec.week_monday(1)).days // 7 + 1 if created else rows[found[0][0]]["week"]
    keep = found[0][0]
    for i, _ in found:
        if abs(rows[i]["week"] - made) < abs(rows[keep]["week"] - made):
            keep = i
    for i, _ in found:
        if i != keep:
            rows[i].update(score=0, review=True, comment="", notes=rows[i]["notes"] + f"; same item as week {rows[keep]['week']}, where it was made: credit once")

revisions = pd.DataFrame(rows, columns=COLUMNS)
path = GRADING_DIR / "revisions-w02-09.csv"
revisions.to_csv(path, index=False)
print("Review and edit:", path)
revisions[revisions["review"]][["week", "name", "score", "notes", "url"]]

## Portfolio checkpoint, weeks 11–14

Each student submits the URL of their public project repository. For each one:
1. Clone it into `GRADING_DIR/repos`.
2. Check out its last commit before the deadline.
3. Check the week's two criteria, at 1 point each.

The checks look for the files and README sections the week asks for, so they are a first pass:
any row under 2 points is flagged for you to look at the repo.

In [ ]:
WEEK = 11

a = assignments[(assignments["group"] == "Textbook Revisions") & (assignments["week"] == WEEK)].iloc[0]
r = canvas.get(f"{CANVAS}/assignments/{a['assignment_id']}/submissions", params={"per_page": 100})
r.raise_for_status()
rows = []
for s in r.json():
    if s["user_id"] not in roster.index:
        continue
    row = {"kind": "portfolio", "week": WEEK, "assignment_id": a["assignment_id"], "user_id": s["user_id"],
           "name": roster.loc[s["user_id"], "name"], "action": "grade", "score": 0, "rubric": "[0, 0]",
           "comment": "", "review": False, "notes": "", "url": s.get("url") or ""}
    m = re.match(r"https?://github\.com/([^/]+)/([^/#?]+)", row["url"])
    if not m:
        row.update(notes="no submission" if not row["url"] else "not a GitHub repository link", review=bool(row["url"]))
        rows.append(row)
        continue
    repo = GRADING_DIR / "repos" / f"{m.group(1)}__{m.group(2)}".lower()
    if repo.exists():
        done = subprocess.run(["git", "-C", repo, "fetch", "-q", "origin"])
    else:
        done = subprocess.run(["git", "clone", "-q", "--filter=blob:none", f"https://github.com/{m.group(1)}/{m.group(2)}", repo])
    sha = subprocess.run(["git", "-C", repo, "rev-list", "-1", f"--before={a['due_at']}", "origin/HEAD"],
                         capture_output=True, text=True).stdout.strip() if done.returncode == 0 else ""
    if not sha:
        row.update(notes="could not clone" if done.returncode else "nothing pushed by the deadline", review=True)
        rows.append(row)
        continue
    subprocess.run(["git", "-C", repo, "checkout", "-q", "--detach", sha])

    files = [p for p in repo.rglob("*") if p.is_file() and ".git" not in p.parts]
    names = {p.name.lower() for p in files}
    readme = next((p for p in files if p.name.lower() == "readme.md" and p.parent == repo), None)
    text = readme.read_text(errors="replace") if readme else ""
    headings = " ".join(h.lower() for h in re.findall(r"^#+\s*(.+)$", text, re.M))
    section_words = {}
    for name in ("data", "method", "finding", "reproduc", "how to"):
        found = re.findall(rf"^#+\s*{name}[^\n]*\n(.*?)(?=^#|\Z)", text, re.M | re.S | re.I)
        section_words[name] = max([len(f.split()) for f in found] + [0])
    notebooks = [p for p in files if p.suffix == ".ipynb"]
    code_files = [p for p in files if p.suffix in (".py", ".ipynb")]
    data_files = [p for p in files if "data" in [x.lower() for x in p.relative_to(repo).parts[:-1]]]
    figures = [p for p in files if p.suffix.lower() in (".png", ".jpg", ".jpeg", ".svg") or (p.suffix.lower() == ".pdf" and p.name.lower() != "report.pdf")]
    report = next((p for p in files if p.name.lower() in ("report.md", "report.pdf")), None)
    code_cells = cells_with_output = 0
    for nb in notebooks:
        try:
            nb_cells = json.loads(nb.read_text(errors="replace")).get("cells", [])
        except ValueError:
            nb_cells = []
        for c in nb_cells:
            if c.get("cell_type") == "code" and "".join(c.get("source", [])).strip():
                code_cells += 1
                cells_with_output += bool(c.get("outputs"))

    if WEEK == 11:
        c1 = bool(text) and len(text.split()) >= 40 and sum(h in headings for h in ("data", "method", "finding", "reproduc")) >= 3
        c2 = "proposal.md" in names and bool(notebooks) and (report is not None or "report" in text.lower()) and (repo / "data").is_dir()
    if WEEK == 12:
        c1 = any(re.search(r"user-agent|headers\s*=", p.read_text(errors="replace"), re.I) for p in code_files if p.stat().st_size < 5_000_000)
        c2 = bool(data_files) and section_words["data"] >= 40
    if WEEK == 13:
        c1 = code_cells > 0 and cells_with_output >= max(1, code_cells // 2)
        c2 = bool(figures) and section_words["method"] >= 20 and section_words["finding"] >= 20
    if WEEK == 14:
        words = len(report.read_text(errors="replace").split()) if report is not None and report.suffix == ".md" else None
        c1 = report is not None and (words is None or words >= 1000)
        c2 = {"readme.md", "proposal.md"} <= names and bool(notebooks) and report is not None and section_words["reproduc"] + section_words["how to"] >= 20
    row.update(score=int(c1) + int(c2), rubric=json.dumps([int(c1), int(c2)]), review=not (c1 and c2),
               notes=f"commit {sha[:7]}; {len(files)} files; README headings: {headings[:80] or 'none'}; "
                     f"data files {len(data_files)}; code cells with output {cells_with_output} of {code_cells}; "
                     f"figures {len(figures)}; report {report.name if report is not None else 'none'}")
    rows.append(row)

portfolio = pd.DataFrame(rows, columns=COLUMNS)
path = GRADING_DIR / f"portfolio-w{WEEK:02d}.csv"
portfolio.to_csv(path, index=False)
print("Review and edit:", path)
portfolio[portfolio["review"]][["name", "score", "notes", "url"]]

## Notebook Labs

Students upload the companion notebook as HTML. First, download every upload and get the companion
notebook as it stood before the due date: the scaffold the students filled in. Some versions of a
notebook in the book were broken JSON, so if the latest one before the due date doesn't parse, step
back to the most recent one that does.

In [ ]:
WEEK = 11

a = assignments[(assignments["group"] == "Notebook Labs") & (assignments["week"] == WEEK)].iloc[0]
lab_dir = GRADING_DIR / "labs" / f"week-{WEEK:02d}"
lab_dir.mkdir(parents=True, exist_ok=True)
url, params, lab_subs = f"{CANVAS}/assignments/{a['assignment_id']}/submissions", {"per_page": 100}, []
while url:
    r = canvas.get(url, params=params)
    r.raise_for_status()
    for s in r.json():
        if s["user_id"] not in roster.index:
            continue
        files = []
        for i, att in enumerate(s.get("attachments") or []):
            p = lab_dir / f"{s['user_id']}_{i}.html.txt"     # .txt, so it's never opened as a web page
            p.write_bytes(canvas.get(att["url"]).content)
            files.append(p)
        lab_subs.append({"user_id": s["user_id"], "submitted_at": s.get("submitted_at"), "files": files})
    url, params = r.links.get("next", {}).get("url"), None

slug = spec.week_info(WEEK)[4]
r = github.get(f"https://api.github.com/repos/cuinfoscience/Web-Data-Science-Book/commits",
               params={"path": f"notebooks/{slug}.ipynb", "until": a["due_at"], "per_page": 10})
scaffold_nb = None
for commit in (r.json() if r.ok else []):
    raw = requests.get(f"https://raw.githubusercontent.com/cuinfoscience/Web-Data-Science-Book/{commit['sha']}/notebooks/{slug}.ipynb")
    try:
        scaffold_nb = raw.json()
        break
    except ValueError:
        continue
print(len(lab_subs), "submissions;", "scaffold from", commit["sha"][:7] if scaffold_nb else "nowhere: check the book")

Next, split the scaffold's Recommended Exercises into steps, the way the students' copies will be
split. Each step keeps its code lines and its words. Later, the scaffold's lines and words are
subtracted from each student's, so what's left is the student's own work. The last step is the
interpretation.

In [ ]:
STEP = re.compile(r"(?:^|\n)\s*(?:#+\s*)?(?:\*\*)?Step\s*(\d+)\s*(?:—|–|-|:|\.|\*\*|¶|$)", re.I | re.M)
END = re.compile(r"^\s*#*\s*(Additional Exercises|Social History|Public Interest|Common Issues|Key Takeaways|Further Reading|Missing Manual Reference)", re.I)

scaffold_code, scaffold_words, step, inside = defaultdict(Counter), defaultdict(Counter), 0, False
for c in scaffold_nb["cells"]:
    src = "".join(c["source"])
    if c["cell_type"] == "markdown" and re.search(r"^\s*#*\s*Recommended Exercises", src, re.M):
        inside = True
    elif inside and c["cell_type"] == "markdown" and END.search(re.sub(r"[*`_#>]", "", src)):
        break
    if not inside:
        continue
    found = [int(x) for x in STEP.findall(re.sub(r"[`_>]", "", src)) if 0 < int(x) < 12]
    if found:
        step = max(found)
    scaffold_code[step].update(l.strip() for l in src.splitlines() if c["cell_type"] == "code" and l.strip() and not l.strip().startswith("#"))
    scaffold_words[step].update(w.lower() for w in re.findall(r"[A-Za-z']{2,}", re.sub(r"[*`_#>]", "", src)))
last_step = max(scaffold_code) if scaffold_code else 0
required = [k for k in range(1, last_step) if k not in OPTIONAL_STEPS.get(WEEK, set())]
print("Steps:", last_step, "| required code steps:", required)

Then check each student's upload against the scaffold. Each upload is read in five stages:
1. Read the HTML into cells.
2. Find the Recommended Exercises section.
3. Assign each cell to a step.
4. Count the student's own code lines in each step, and the new words in the interpretation, which
   can be in the prompt cell, comments, Markdown, or a `print()`.
5. Note unrun cells and tracebacks.

When a student uploaded two files, the fuller one is used.

In [ ]:
checks = {}
for s in lab_subs:
    best = None
    for p in s["files"]:
        soup = BeautifulSoup(p.read_text(errors="replace"), "lxml")
        cells = []
        for c in soup.select("div.jp-Cell") or soup.select("div.cell"):
            classes = c.get("class", [])
            if "jp-MarkdownCell" in classes or "text_cell" in classes:
                cells.append({"md": True, "text": c.get_text("\n", strip=True)})
            elif "jp-CodeCell" in classes or "code_cell" in classes:
                editor = c.select_one(".jp-InputArea-editor, .input_area")
                outputs = c.select(".jp-OutputArea-output, .output_area")
                prompt = c.select_one(".jp-InputPrompt, .input_prompt")
                out_text = "\n".join(o.get_text("\n", strip=True) for o in outputs)
                cells.append({"md": False, "text": editor.get_text() if editor else "", "outputs": len(outputs),
                              "ran": bool(prompt and re.search(r"\d", prompt.get_text())),
                              "traceback": bool(re.search(r"Traceback \(most recent call last\)|^\w+(Error|Exception):", out_text, re.M))})
        start = next((i for i, c in enumerate(cells) if c["md"] and re.search(r"^\s*#*\s*Recommended Exercises", c["text"], re.M)), None)
        if start is None:
            start = next((i for i, c in enumerate(cells) if re.search(r"Step\s*1\b", c["text"])), None)
        result = {"file": p.name, "found": start is not None and bool(cells), "attempted": [], "interp": 0,
                  "code_cells": 0, "unrun": 0, "traceback": False, "outputs": 0, "numbers": False, "ai_signs": False,
                  "notes": [] if cells else ["not a Jupyter HTML export"]}
        if result["found"]:
            section = []
            for c in cells[start:]:
                if section and c["md"] and END.search(c["text"]):
                    break
                section.append(c)
            code, words, step = defaultdict(Counter), defaultdict(Counter), 0
            for c in section:
                found = [int(x) for x in STEP.findall(c["text"]) if 0 < int(x) < 12]
                if found:
                    step = max(found)
                if not c["md"]:
                    code[step].update(l.strip() for l in c["text"].splitlines() if l.strip() and not l.strip().startswith("#"))
                words[step].update(w.lower() for w in re.findall(r"[A-Za-z']{2,}", c["text"]))
                if not c["md"] and any(l.strip() and not l.strip().startswith("#") for l in c["text"].splitlines()):
                    result["code_cells"] += 1
                    result["unrun"] += not c["ran"]
                    result["traceback"] |= c["traceback"]
                    result["outputs"] += c["outputs"]
            result["attempted"] = [k for k in required if sum((code[k] - scaffold_code[k]).values()) > 0]
            result["interp"] = sum((words[last_step] - scaffold_words[last_step]).values())
            if result["interp"] < 3:      # an answer under its own heading, such as "Final interpretation"
                extra = Counter(w.lower() for c in section if c["md"] and re.match(r"\s*(final\s+)?interpret", c["text"], re.I)
                                for w in re.findall(r"[A-Za-z']{2,}", c["text"]))
                result["interp"] = max(result["interp"], sum((extra - scaffold_words[last_step]).values()))
            result["numbers"] = bool(re.search(r"\d{2,}", " ".join(c["text"] for c in section if c["md"]))) and result["outputs"] == 0
            result["ai_signs"] = bool(re.search(r"revise it to reflect your own|no measurements have been invented|example interpretation", " ".join(c["text"] for c in section), re.I))
        elif cells:
            result["notes"].append("no Recommended Exercises section or step labels")
        if best is None or (len(result["attempted"]), result["interp"]) > (len(best["attempted"]), best["interp"]):
            best = result
    if best and len(s["files"]) > 1:
        best["notes"].append(f"{len(s['files'])} files; checked the fullest")
    checks[s["user_id"]] = best
pd.DataFrame([{"name": roster.loc[u, "name"], **{k: v for k, v in c.items() if k != "notes"}} for u, c in checks.items() if c])

Finally, the scores. Criteria 1–4 of the rubric come from the checks:

| Criterion | 2 | 1 | 0 |
|---|---|---|---|
| 1. Valid file | any HTML | | |
| 2. Required steps | every required step | at least half | fewer than half |
| 3. Run | every cell run | some unrun | none run |
| 4. Interpretation | 50+ new words | 3–49 | fewer |

Before week 11, unrun code got full credit on criterion 3 (`UNRUN_FULL_CREDIT`).

Criterion 5, whether each step does its task, starts at 2. Only a reader can judge it: lower it in
the CSV where needed.

Then:
- **Blank scaffold:** 0, with the reminder.
- **Late work:** after 24 hours' grace, 1 point for every started day, at most 5, with the comment.
- **Comments** for unrun code and tracebacks.
- **Flagged for review:** a missing step, a thin interpretation, numbers in an interpretation with
  no output to back them, two files, an unusual file, and phrases that look AI-generated.

A file the checks can't read has no score: score it by hand.

In [ ]:
UNRUN_FULL_CREDIT = WEEK < 11

rows = []
for s in lab_subs:
    row = {"kind": "lab", "week": WEEK, "assignment_id": a["assignment_id"], "user_id": s["user_id"],
           "name": roster.loc[s["user_id"], "name"], "action": "grade", "score": 0, "rubric": "",
           "comment": "", "review": False, "notes": "", "url": ""}
    c = checks.get(s["user_id"])
    if not s["files"] or c is None:
        row["notes"] = "no submission"
        rows.append(row)
        continue
    if not c["found"]:
        row.update(score="", review=True, notes="; ".join(c["notes"] + ["score by hand"]))
        rows.append(row)
        continue
    missing = [k for k in required if k not in c["attempted"]]
    e2 = 2 if not missing else 1 if len(c["attempted"]) >= len(required) / 2 else 0
    e3 = (2 if c["code_cells"] else 0) if (UNRUN_FULL_CREDIT or c["unrun"] == 0) else (1 if c["unrun"] < c["code_cells"] else 0)
    e4 = 2 if c["interp"] >= 50 else 1 if c["interp"] >= 3 else 0
    points = [2, e2, e3, e4, 2]
    comments = []
    if not c["attempted"] and c["interp"] < 10:
        score, comments = 0, [COMMENTS["blank"]]
        row["review"] = True
    else:
        score = sum(points)
        if c["numbers"]:
            comments.append(COMMENTS["unseen"])
            row["review"] = True
        elif c["unrun"]:
            comments.append(COMMENTS["unrun"])
        if c["traceback"]:
            comments.append(COMMENTS["traceback"])
        hours = (datetime.fromisoformat(s["submitted_at"].replace("Z", "+00:00")) - datetime.fromisoformat(a["due_at"].replace("Z", "+00:00"))).total_seconds() / 3600
        days = math.ceil(hours / 24) if hours > LATE_GRACE_HOURS else 0
        if days:
            deducted = min(days, LATE_CAP)
            score = max(0, score - deducted)
            comments.append(f"This lab was submitted {days} day{'s' if days > 1 else ''} late, past the 24-hour grace period, "
                            f"so {deducted} point{'s were' if deducted > 1 else ' was'} deducted"
                            + (" (the late penalty is capped at 5 points)." if days > LATE_CAP else "."))
    notes = [f"steps missing: {missing}" if missing else "all required steps", f"{c['unrun']} of {c['code_cells']} code cells unrun",
             f"interpretation ~{c['interp']} new words"] + c["notes"]
    if c["numbers"]:
        notes.append("interpretation cites numbers but nothing ran")
    if c["ai_signs"]:
        notes.append("phrases that look AI-generated: hold for the instructor?")
    row.update(score=score, rubric=json.dumps(points) if WEEK >= 11 else "", comment=" ".join(comments), notes="; ".join(notes),
               review=row["review"] or bool(missing) or e4 == 1 or c["ai_signs"] or bool(c["notes"]))
    rows.append(row)

labs = pd.DataFrame(rows, columns=COLUMNS)
path = GRADING_DIR / f"labs-w{WEEK:02d}.csv"
labs.to_csv(path, index=False)
print("Review and edit:", path)
labs[labs["review"]][["name", "score", "notes"]]

## Commendations

A short comment on particularly strong work, at most two or three per student across the semester,
and only for work whose code and output are in the submission. Comments only, with no grade. Add a
row for each, then post `commendations.csv` below.

In [ ]:
commendations = pd.DataFrame([
    # {"kind": "lab", "week": 11, "assignment_id": 2852043, "user_id": 123456, "name": "...", "action": "comment_only",
    #  "score": "", "rubric": "", "comment": "Great find that ...", "review": False, "notes": "", "url": ""},
], columns=COLUMNS)
commendations.to_csv(GRADING_DIR / "commendations.csv", index=False)

## Post a reviewed table

Set `TABLE` to the CSV you reviewed. Posting:
1. Only rows whose `action` is `grade` or `comment_only` are posted; `hold` and `skip` are left alone.
2. A row with rubric points fills in the assignment's Canvas rubric. If its score differs from the
   rubric total (a late penalty or a blank scaffold), the score is posted after the rubric.
3. Every row is read back from Canvas. Any row that doesn't match is listed.

Nothing is sent until `APPLY = True` and `CONFIRM` matches the number of rows.

In [ ]:
TABLE = "labs-w11"
CONFIRM = ""           # type "post N", where N is the number of rows printed below

table = pd.read_csv(GRADING_DIR / f"{TABLE}.csv", dtype=str, keep_default_na=False)
table["review"] = table["review"].str.lower().isin(["true", "1", "yes"])
to_post = table[table["action"].isin(["grade", "comment_only"])]
unscored = to_post[(to_post["action"] == "grade") & (to_post["score"] == "")]
print(len(to_post), "rows to post;", int(to_post["review"].sum()), "still marked for review;", len(unscored), "with no score")

if to_post["review"].any() or len(unscored):
    print("Resolve the rows still marked for review, and fill in every score, first.")
elif not APPLY:
    print("Dry run: set APPLY = True in Setup to post.")
elif CONFIRM != f"post {len(to_post)}":
    print(f'Set CONFIRM = "post {len(to_post)}" to post these rows.')
else:
    criteria = {}
    for _, row in to_post.iterrows():
        target = f"{CANVAS}/assignments/{row['assignment_id']}/submissions/{row['user_id']}"
        data = {"comment[text_comment]": row["comment"]} if row["comment"] else {}
        if row["action"] == "grade" and row["rubric"]:
            if row["assignment_id"] not in criteria:
                criteria[row["assignment_id"]] = [c["id"] for c in canvas.get(f"{CANVAS}/assignments/{row['assignment_id']}").json().get("rubric") or []]
            points = json.loads(row["rubric"])
            if len(points) != len(criteria[row["assignment_id"]]):
                raise ValueError(f"{row['name']}: the rubric has {len(criteria[row['assignment_id']])} criteria and the row has {len(points)}")
            for criterion, p in zip(criteria[row["assignment_id"]], points):
                data[f"rubric_assessment[{criterion}][points]"] = p
        elif row["action"] == "grade":
            data["submission[posted_grade]"] = row["score"]
        canvas.put(target, data=data).raise_for_status()
        if row["action"] == "grade" and row["rubric"] and float(row["score"]) != sum(json.loads(row["rubric"])):
            canvas.put(target, data={"submission[posted_grade]": row["score"]}).raise_for_status()
    print("Posted", len(to_post), "rows")

Read the table back from Canvas. A `hold` row should still have no score. Run this any time to
check a table you posted earlier.

In [ ]:
mismatches, live = [], {}
for assignment_id in sorted(set(table["assignment_id"])):
    url, params = f"{CANVAS}/assignments/{assignment_id}/submissions", {"include[]": "submission_comments", "per_page": 100}
    while url:
        r = canvas.get(url, params=params)
        r.raise_for_status()
        for s in r.json():
            live[(str(assignment_id), str(s["user_id"]))] = s
        url, params = r.links.get("next", {}).get("url"), None
for _, row in table[table["action"].isin(["grade", "comment_only", "hold"])].iterrows():
    s = live.get((row["assignment_id"], row["user_id"]), {})
    ok = not row["comment"] or row["comment"] in [c["comment"] for c in s.get("submission_comments") or []]
    if row["action"] == "grade":
        ok = ok and row["score"] != "" and s.get("score") is not None and float(s["score"]) == float(row["score"])
    if row["action"] == "hold":
        ok = s.get("score") is None
    if not ok:
        mismatches.append({"name": row["name"], "week": row["week"], "action": row["action"], "wanted": row["score"], "canvas": s.get("score")})
print(len(mismatches), "rows don't match Canvas")
pd.DataFrame(mismatches)